# 自定义中间件-Wrap-style hooks

## 本节导读

上一节的 Node-style hooks 在 Agent 执行图的固定节点上运行；本节的 **Wrap-style hooks** 则会把某一次模型调用或工具调用“包裹”起来。它既能在调用前处理请求，也能在调用后处理结果，还能直接决定被包裹的调用是否执行、执行一次还是执行多次。

Wrap-style 最核心的参数是 `handler`：

```text
进入中间件
    ↓
调用前逻辑 → handler(request) → 调用后逻辑
                       ↓
                 真正的模型/工具
```

- **不调用 `handler`**：短路本次调用，直接返回缓存或兜底结果。
- **调用一次 `handler`**：正常执行，可在前后做参数改写、日志、审计或结果转换。
- **调用多次 `handler`**：可实现重试、降级或多次采样，但必须考虑调用成本和副作用。

### Wrap-style 与 Node-style 的区别

| 对比项 | Node-style hooks | Wrap-style hooks |
|---|---|---|
| 执行方式 | 在固定生命周期节点顺序执行 | 包围一次具体的模型或工具调用 |
| 是否拿到 `handler` | 否 | 是 |
| 对底层调用的控制 | 通过状态更新或 `jump_to` 间接控制 | 可决定 `handler` 调用 0、1 或多次 |
| 常见场景 | 状态更新、校验、阶段日志 | 重试、缓存、降级、参数改写、异常转换 |

### 本节包含的两类 Hook

| Hook | 包裹对象 | 主要输入 | 主要返回值 |
|---|---|---|---|
| `wrap_model_call` | 每一次模型调用 | `ModelRequest` 和模型 `handler` | `ModelResponse` |
| `wrap_tool_call` | 每一次工具调用 | `ToolCallRequest` 和工具 `handler` | `ToolMessage` 或 `Command` |

两类 Hook 都支持基于装饰器和基于类的实现。简单、独立的逻辑可使用装饰器；需要参数化配置、复用或组合多个 Hook 时，更适合继承 `AgentMiddleware` 实现类式中间件。

> **重要原则**：正常情况下应确保 `handler` 被调用一次并返回它的结果。如果有意调用多次，要设置明确的次数上限。对“发消息、扣款、写数据库”等有副作用的工具，重复调用 `handler` 可能会重复执行真实操作。

## 1、wrap_model_call 的使用

`wrap_model_call` 会拦截每一次模型请求。本节示例在调用 `handler(request)` 前修改最后一条输入消息，并在模型返回后修改响应内容，用于直观观察“调用前 + 真正调用 + 调用后”的完整过程。

### 1.1 基于装饰器的实现

In [ ]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# 从.env文件中加载环境变量
load_dotenv(override=True)

model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL")
)

In [2]:

from typing import Callable
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse, AgentMiddleware


@wrap_model_call
def wrap_model_call_middleware(
        request: ModelRequest,
        handler: Callable[[ModelRequest], ModelResponse],
) -> ModelResponse | None:
    request.messages[-1].content += "---> wrap_model_call_before <---"

    # 模型的调用
    response = handler(request)

    response.result[0].content += "---> wrap_model_call_after <---"

    return response

In [3]:

from langchain_core.messages import HumanMessage
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    middleware=[
        wrap_model_call_middleware,
    ]
)

response = agent.invoke({
    "messages": [HumanMessage("你好")]
})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

你好---> wrap_model_call_before <---
================================== Ai Message ==================================

你好！你是想让我帮你处理 `wrap_model_call_before` 相关的内容吗？  
如果你是在测试标签/占位符，我也可以直接继续配合。---> wrap_model_call_after <---


### 1.2 基于类的实现

In [4]:
from langchain.agents.middleware import AgentMiddleware


class WrapModelCallMiddleware(AgentMiddleware):
    def wrap_model_call(self, request: ModelRequest,
                        handler: Callable[[ModelRequest], ModelResponse],
                        ) -> ModelResponse | None:
        request.messages[-1].content += "---> wrap_model_call_before <---"

        # 模型的调用
        response = handler(request)

        response.result[0].content += "---> wrap_model_call_after <---"

        return response


agent = create_agent(
    model=model,
    middleware=[
        WrapModelCallMiddleware(),
    ]
)

response = agent.invoke({
    "messages": [HumanMessage("你好")]
})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

你好---> wrap_model_call_before <---
================================== Ai Message ==================================

你好！有什么我可以帮你的吗？---> wrap_model_call_after <---


## 2、wrap_tool_call 的使用

`wrap_tool_call` 会拦截 Agent 发起的每一次工具调用。可在执行前检查或改写 `request.tool_call["args"]`，也可捕获异常、记录耗时，或将工具结果转换为统一的 `ToolMessage`/`Command`。

本节示例会先按模型生成的原始参数调用一次天气工具，再将 `is_forcast` 改为 `True` 后调用第二次，最终返回第二次结果。这是为了演示中间件可完全控制 `handler`；实际项目中，如果只是想修正参数，通常应该先修改参数，然后只调用一次 `handler`。

### 2.1 基于装饰器的实现

In [5]:

from langchain_core.tools import tool
from typing import Any
from langgraph.types import Command
from langchain_core.messages import ToolMessage
from langgraph.prebuilt.tool_node import ToolCallRequest
from langchain.agents.middleware import wrap_tool_call


@tool
def get_weather(city: str, is_forcast: bool) -> str:
    """
    获取当日特定城市的天气

    Args:
        city: 城市名称
        is_forcast: 是否包含明天的天气预报
    """
    res = f"{city}今天天气不错"
    if is_forcast:
        res += "\n明天天气也很好"
    return res


@wrap_tool_call
def wrap_tool_call_middleware(request: ToolCallRequest,
                              handler: Callable[[ToolCallRequest], ToolMessage | Command[Any]],
                              ) -> ToolMessage | Command[Any]:
    result = handler(request)
    print(f"原始参数：{request.tool_call['args']}")
    print(f"原始参数调用结果：{result}")

    request.tool_call["args"]["is_forcast"] = True
    result = handler(request)

    print(f"更新以后的参数：{request.tool_call['args']}")
    print(f"更新以后的参数调用结果：{result}")
    return result


agent = create_agent(
    model=model,
    tools=[get_weather],
    middleware=[wrap_tool_call_middleware]
)

response = agent.invoke({
    "messages": [HumanMessage("帮我查询北京今天的天气如何？")]
})

for msg in response["messages"]:
    msg.pretty_print()

原始参数：{'city': '北京', 'is_forcast': False}
原始参数调用结果：content='北京今天天气不错' name='get_weather' tool_call_id='call_MCqn74Cp1iSMbr7mdh9ILS2A'
更新以后的参数：{'city': '北京', 'is_forcast': True}
更新以后的参数调用结果：content='北京今天天气不错\n明天天气也很好' name='get_weather' tool_call_id='call_MCqn74Cp1iSMbr7mdh9ILS2A'
================================ Human Message =================================

帮我查询北京今天的天气如何？
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_MCqn74Cp1iSMbr7mdh9ILS2A)
 Call ID: call_MCqn74Cp1iSMbr7mdh9ILS2A
  Args:
    city: 北京
    is_forcast: True
================================= Tool Message =================================
Name: get_weather

北京今天天气不错
明天天气也很好
================================== Ai Message ==================================

北京今天天气不错。  
如果你需要，我也可以继续帮你整理成更适合出行的建议，比如是否适合带伞、穿什么衣服。


### 2.2 基于类的实现

In [6]:

class WrapToolCallMiddleware(AgentMiddleware):
    def wrap_tool_call(self, request: ToolCallRequest,
                       handler: Callable[[ToolCallRequest], ToolMessage | Command[Any]],
                       ) -> ToolMessage | Command[Any]:
        result = handler(request)
        print(f"原始参数：{request.tool_call['args']}")
        print(f"原始参数调用结果：{result}")

        request.tool_call["args"]["is_forcast"] = True
        result = handler(request)

        print(f"更新以后的参数：{request.tool_call['args']}")
        print(f"更新以后的参数调用结果：{result}")
        return result


agent = create_agent(
    model=model,
    tools=[get_weather],
    middleware=[WrapToolCallMiddleware()]
)

response = agent.invoke({
    "messages": [HumanMessage("帮我查询上海今天的天气如何？")]
})

for msg in response["messages"]:
    msg.pretty_print()

原始参数：{'city': '上海', 'is_forcast': False}
原始参数调用结果：content='上海今天天气不错' name='get_weather' tool_call_id='call_m2ft9YtgbGaWqXs5q2J6aB7D'
更新以后的参数：{'city': '上海', 'is_forcast': True}
更新以后的参数调用结果：content='上海今天天气不错\n明天天气也很好' name='get_weather' tool_call_id='call_m2ft9YtgbGaWqXs5q2J6aB7D'
================================ Human Message =================================

帮我查询上海今天的天气如何？
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_m2ft9YtgbGaWqXs5q2J6aB7D)
 Call ID: call_m2ft9YtgbGaWqXs5q2J6aB7D
  Args:
    city: 上海
    is_forcast: True
================================= Tool Message =================================
Name: get_weather

上海今天天气不错
明天天气也很好
================================== Ai Message ==================================

上海今天天气不错，适合外出。  
如果你需要，我也可以继续帮你看一下明天的天气建议穿什么。
